# Unsupervised Physics-Loss Fine-Tuning

This notebook demonstrates fine-tuning an already-pretrained phase reconstructor using **only the WFS's own detector frames**, never the phase that produced them -- the situation any deployed AO system is actually in, since a real bench has no independent way to measure the true OPD behind a given image. The technique, its derivation, and its risks are written up in `Ideas/08-unsupervised-physics-finetuning.md`; this notebook builds the demo it describes.

Two phases, on the same reconstructor:

- **Phase A (supervised warm start)** -- ordinary training via `AI4AO.Trainer.Trainer`, with ground-truth OPD available, exactly like `Tutorials/basics/05_TrainingAReconstructor.ipynb`. Deliberately stopped short of convergence, so Phase B has visible room to improve.
- **Phase B (unsupervised fine-tune)** -- the same `Trainer`, the same on-the-fly `PhaseDataset` stream, but with `trainer.loss` swapped to a bare `AI4AO.LossFunctions.Physics_loss`. WFS frames are still generated fresh every step -- an endless synthetic stream, exactly like Phase A and every other notebook in this repo. The difference between the two phases is entirely in which loss is plugged in, not in how the data is produced.

Ground-truth OPD is only ever used twice outside of Phase A: to physically generate each WFS frame (unavoidable -- a real detector frame also comes from a real, equally unlabeled wavefront), and at the very end, to check whether the fine-tune actually helped.

In [ ]:
from mmengine import Config
import copy
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

from AI4AO import PyramidWFS, PhaseDataset, FramePreprocess, DeformableMirror, Trainer, imshow_multiple
from AI4AO.LossFunctions import LogResidualVarianceLoss, Physics_loss

device = 'cuda' if torch.cuda.is_available() else 'cpu'

paramfile = 'UnsupervisedFineTuning_params.py'

AtmosParams = Config.fromfile(paramfile)['AtmosParams']
WFSParams = Config.fromfile(paramfile)['WFSParams']
LoopParams = Config.fromfile(paramfile)['LoopParams']
DMParams = Config.fromfile(paramfile)['DMParams']
TrainParams = Config.fromfile(paramfile)['TrainParams']

## Building the twin, dataset, and frame preprocessor

This is a fresh fictional demo instrument (own params file, no real bench behind it, no `LoadCalibration`) -- both `wfs` and `dm` are frozen `.eval()` throughout this notebook, since only the reconstructor's weights are ever trained, in either phase. `dataset.generateClosedLoop = True` matches `05`'s own setup; it only matters once `closed_loop_iterations > 1` is tried (see the closing notes), since with `closed_loop_iterations = 1` every training step is a single independent open-loop draw either way.

In [ ]:
wfs = PyramidWFS(WFSParams, device)
dm = DeformableMirror(WFSParams, DMParams, device)
wfs.eval()
dm.eval()

framePreprocessor = FramePreprocess(WFSParams, wfs, device)
framePreprocessor.ProcessReference(wfs.reference_intensity)

M2C = dm.MakeZernikeM2C()

dataset = PhaseDataset(WFSParams, AtmosParams, LoopParams, DMParams, device)
dataset.generateClosedLoop = True

## Reconstructor architecture

`PWFSNet`, copied verbatim from `05_TrainingAReconstructor.ipynb`: a grouped-conv stem (each of the Pyramid's 4 pupil images is processed independently at first), a shared encoder, and a linear head onto `Nmodes` actuator coefficients.

In [ ]:
class PWFSNet(nn.Module):
    def __init__(self, DMParams):
        super().__init__()

        Nmodes = DMParams["Nmodes"]

        self.stem = nn.Sequential(
            # Process each pupil independently
            nn.Conv2d(4, 32, kernel_size=11, padding=5, groups=4),
            nn.GELU(),

            nn.Conv2d(32, 64, kernel_size=7, padding=3, groups=4),
            nn.GELU(),

            nn.MaxPool2d(2),      # 42 -> 21
        )

        self.encoder = nn.Sequential(
            nn.Conv2d(64, 64, 5, padding=2),
            nn.GELU(),

            nn.MaxPool2d(2),      # 21 -> 10

            nn.Conv2d(64, 128, 3, padding=1),
            nn.GELU(),

            nn.MaxPool2d(2),      # 10 -> 5

            nn.Conv2d(128, 256, 3, padding=1),
            nn.GELU(),

            nn.MaxPool2d(2),      # 10 -> 5

            nn.Conv2d(256, 512, 2, padding=1),
            nn.GELU(),

            nn.AdaptiveAvgPool2d(1)
        )

        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(512, Nmodes)
        )

    def forward(self, x):
        x = self.stem(x)
        x = self.encoder(x)
        return self.head(x)


phaseReconstructor = PWFSNet(DMParams).to(device=device)

total_params = sum(p.numel() for p in phaseReconstructor.parameters() if p.requires_grad)
print(f"Total trainable parameters: {total_params:,}")

## Phase A -- supervised warm start

Ordinary training via `Trainer.train()`, exactly like `05_TrainingAReconstructor.ipynb`: `LogResidualVarianceLoss + Physics_loss` against the dataset's actual ground-truth OPD. `WarmStartSteps` is deliberately picked short of `05`'s own `TrainRunNb = 1000` default -- enough to get the reconstructor off its random initialization, but not so much that Phase B has no headroom left to visibly improve (see `Ideas/08-unsupervised-physics-finetuning.md`, "Known risks" -- an undertrained-enough Phase A is what makes the before/after comparison at the end of this notebook meaningful rather than a flat line).

In [ ]:
warmstart_loss = LogResidualVarianceLoss(dataset.pupil, wfs.wavelength) + Physics_loss(wfs=wfs)
optimizer = torch.optim.AdamW(phaseReconstructor.parameters(), TrainParams['lrn_warmstart'], fused=(device == 'cuda'))

trainer = Trainer(
    wfs=wfs,
    framePreprocessor=framePreprocessor,
    dm=dm,
    M2C=M2C,
    phaseReconstructor=phaseReconstructor,
    dataset=dataset,
    loss=warmstart_loss,
    optimizer=optimizer,
)

loss_tracker_a, loss_tracker_ideal_a = trainer.train(
    TrainParams['WarmStartSteps'], TrainParams['closed_loop_iterations']
)

In [ ]:
trainer.plot_losses(loss_tracker_a, loss_tracker_ideal_a)

In [ ]:
# Snapshot Phase A's weights before Phase B mutates `phaseReconstructor` further --
# this is the "before" checkpoint for the honesty check at the end of the notebook.
phase_a_state = copy.deepcopy(phaseReconstructor.state_dict())

## Why swapping the loss is enough to make Phase B unsupervised

`Trainer.train()`'s inner loop always draws a fresh ground-truth OPD (`opd_gt`) from `dataset` and uses it to physically form the WFS frame -- that part never changes between phases, and it shouldn't: a real detector frame also comes from a real, equally unlabeled wavefront. What matters is what reaches the *loss* that gets backpropagated. With `closed_loop_iterations = 1`:

- `residual_opd = opd_gt` (no correction applied yet), so `wfs_frames = wfs(opd_gt, pupil)` is the physically observed frame.
- `z_output = phaseReconstructor(preprocessed_frames)` never sees `opd_gt` -- it only ever sees the WFS frame.
- `corrected_residual_opd = residual_opd - dm(z_output @ M2C.T)`.

`Physics_loss.compute()` (`AI4AO/LossFunctions.py:138-150`) computes `reconstructed_opd = residual_opd - corrected_residual_opd`, which algebraically equals `dm(z_output @ M2C.T)` -- **the network's own predicted OPD, with every `opd_gt` term cancelling out exactly.** Its returned scalar is a function of the network's prediction and the observed frame only. `Trainer.train()` also computes `Ze`/`modes`/`ideal_loss` from `opd_gt`, but strictly under `torch.no_grad()` or only inside `ideal_loss` (which is tracked for reference, never backpropagated) -- so as long as `trainer.loss` is a **bare** `Physics_loss`, with no other term mixed in, nothing derived from `opd_gt` ever reaches the optimizer step.

This is why Phase B below doesn't need a new training loop at all -- just a different `loss` plugged into the same `trainer`. The one thing that must not happen is accidentally reusing Phase A's combined loss (`LogResidualVarianceLoss + Physics_loss`) for Phase B: `LogResidualVarianceLoss` *does* need the true residual phase variance, so leaving it in would silently turn Phase B back into supervised training with no error raised. The `assert` in the next cell exists specifically to catch that mistake.

In [ ]:
trainer.loss = Physics_loss(wfs=wfs)
assert isinstance(trainer.loss, Physics_loss), (
    "Phase B must train against a bare Physics_loss -- combining it with any "
    "ground-truth-dependent term (e.g. LogResidualVarianceLoss) would silently "
    "make this phase supervised again."
)

trainer.optimizer = torch.optim.AdamW(phaseReconstructor.parameters(), TrainParams['lrn_finetune'], fused=(device == 'cuda'))

## Phase B -- unsupervised physics-loss fine-tune

Same `trainer`, same `dataset` object, same on-the-fly frame generation `Trainer.train()` always does -- the only thing that changed is `trainer.loss`/`trainer.optimizer`. `loss_tracker_b` is the real training signal (physics consistency only); `loss_tracker_ideal_b` is the same physics loss evaluated at the oracle correction `Ze` -- a diagnostic lower bound `Trainer` computes for free, but `Ze` never reaches the optimizer (see the derivation above), so it's shown here for reference only, not as evidence the network could see it.

In [ ]:
loss_tracker_b, loss_tracker_ideal_b = trainer.train(
    TrainParams['FineTuneSteps'], TrainParams['closed_loop_iterations']
)

In [ ]:
trainer.plot_losses(loss_tracker_b, loss_tracker_ideal_b)

## Validation -- did the unsupervised fine-tune actually help?

A decreasing physics-consistency loss in Phase B is not, by itself, evidence that phase estimation improved -- it only shows the network got better at reproducing the WFS frames it saw, which a purely noise-fitting solution could also do to some extent. The only honest check is against the ground truth this notebook has otherwise not used since Phase A: run a **seeded, paired** closed-loop rollout (`trainer.evaluate()`, no gradients) with the Phase-A-only weights and again with the Phase-B-fine-tuned weights, on identical atmosphere draws, and compare steady-state residual wavefront error in nm RMS (excluding the leaky-integrator warm-up, `i > n_steps * 0.3`, the same convention `05_TrainingAReconstructor.ipynb` uses).

In [ ]:
def residual_rms_nm(result, pupil_mask, warmup_frac=0.3):
    """Steady-state residual wavefront error (nm RMS), excluding the leaky-integrator warm-up."""
    n_steps = result.residual_opd.shape[0]
    start = int(n_steps * warmup_frac)
    residual = result.residual_opd[start:]
    per_step_rms = torch.sqrt(torch.mean(residual[..., pupil_mask] ** 2, dim=-1))
    return (per_step_rms.mean() * 1e9).item()


phaseReconstructor_before = PWFSNet(DMParams).to(device=device)
phaseReconstructor_before.load_state_dict(phase_a_state)
phaseReconstructor_before.eval()

seed = 1234
pupil_mask = wfs.pupil.bool()

trainer.phaseReconstructor = phaseReconstructor_before
torch.manual_seed(seed)
result_before = trainer.evaluate(n_steps=TrainParams['TestRunNb'])

trainer.phaseReconstructor = phaseReconstructor  # Phase-B-fine-tuned weights
torch.manual_seed(seed)
result_after = trainer.evaluate(n_steps=TrainParams['TestRunNb'])

rms_before = residual_rms_nm(result_before, pupil_mask)
rms_after = residual_rms_nm(result_after, pupil_mask)

print(f"Residual WFE, Phase A only  : {rms_before:8.1f} nm RMS")
print(f"Residual WFE, after Phase B : {rms_after:8.1f} nm RMS")

In [ ]:
from AI4AO import imshow
imshow(result_after.residual_opd[0])
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(4, 4))
ax.bar(["Phase A only", "After Phase B\n(unsupervised)"], [rms_before, rms_after], color=["tab:gray", "tab:blue"])
ax.set_ylabel("Residual WFE (nm RMS)")
ax.set_title("Effect of unsupervised physics-loss fine-tuning")
plt.show()

In [ ]:
import matplotlib as mpl
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

n_frames = 60
result = trainer.evaluate(n_steps=n_frames, dataset=dataset)

fig, axes = imshow_multiple(
    [
        {"tensor": result_before.opd[0], "title": "Input opd", "same_scale": True},
        {"tensor": result_before.residual_opd[0], "title": "Residual opd", "scale_reference": result_before.opd[0]},
        {"tensor": result_after.residual_opd[0], "title": "Residual opd", "scale_reference": result_before.opd[0]},
        {"tensor": torch.sqrt(result_before.psfs[0]), "title": "PSF", "same_scale": True},
        {"tensor": torch.sqrt(result_after.psfs[0]), "title": "PSF", "same_scale": True},
    ],
    max_channel_number = 9
)


def update(i):
    imshow_multiple(
        [
            {"tensor": result_before.opd[i], "title": "Input opd", "same_scale": True},
            {"tensor": result_before.residual_opd[i], "title": "Residual opd", "scale_reference": result_before.opd[i]},
            {"tensor": result_after.residual_opd[i], "title": "Residual opd", "scale_reference": result_before.opd[i]},
            {"tensor": torch.sqrt(result_before.psfs[i]), "title": "PSF", "same_scale": True},
            {"tensor": torch.sqrt(result_after.psfs[i]), "title": "PSF", "same_scale": True},
        ],
        fig=fig, axes=axes, 
        max_channel_number = 9
    )
    return [ax.images[0] for tensor_axes in axes for ax in tensor_axes]


anim = FuncAnimation(fig, update, frames=n_frames, interval=50, blit=False)
plt.close(fig)

mpl.rcParams["animation.embed_limit"] = 200
HTML(anim.to_jshtml())

## Known caveats -- summary

- **Identifiability**: matching a WFS frame doesn't uniquely determine the wavefront (noise, weakly-sensed spatial frequencies, possible sign/degenerate-mode ambiguities). Phase B relies on Phase A's warm start to stay in a good basin -- don't skip Phase A or start Phase B from a near-random network and expect this to work.
- **Phase A must stay deliberately undertrained**, or the before/after comparison above will show no headroom to improve, making the demo look inconclusive rather than validating the technique.
- **`trainer.loss` must be a bare `Physics_loss` during Phase B** -- the `assert` above guards this, but it's worth restating as the single most important thing to get right in this notebook.
- Frames are generated on the fly, forever, from `PhaseDataset`, exactly like every other notebook in this repo -- there is no finite "captured" batch here, so the usual "infinite synthetic stream, no held-out split needed" reasoning applies unchanged.
- See `Ideas/08-unsupervised-physics-finetuning.md` for the full derivation, the case for applying this to a real, `TwinCalibrator`-calibrated instrument instead (swap this notebook's fresh `wfs`/`dm` for `TwinCalibrator.load(instrument_name)`'s calibrated twin, and Phase B's synthetic frames for real recorded ones), and why `closed_loop_iterations > 1` (closed-loop, temporally-correlated unsupervised fine-tuning) works with zero additional code.

If you edit this notebook, remember to clear its outputs before committing -- this is not done automatically.